# 01 — Exploratory Data Analysis

Goal: understand the LinkedIn job-postings corpus and the structured skills we extract from it before training the recommender.

Sections:
1. Corpus shape & missingness
2. Title / experience-level / work-type distributions
3. Skill extraction sanity check
4. Skill co-occurrence preview
5. Role-level aggregation

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

## 1. Corpus shape & missingness

In [ ]:
from src.data_loader import load_raw
raw = load_raw()
print('Shape:', raw.shape)
raw.head(3)

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False) * 100
missing.plot.barh(figsize=(8, 4), title='% missing per column')
plt.gca().invert_yaxis()
plt.xlabel('% missing'); plt.tight_layout(); plt.show()

**Key observation**: `skills_desc` is missing for ~98% of rows. Therefore our skill-extraction pipeline must work primarily off the `description` free-text column, using a curated vocabulary.

## 2. Title / experience-level / work-type distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
raw['formatted_experience_level'].value_counts(dropna=False).plot.bar(ax=axes[0], title='Experience level')
raw['formatted_work_type'].value_counts().plot.bar(ax=axes[1], title='Work type')
for ax in axes: ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
raw['title'].value_counts().head(20).plot.barh(figsize=(8, 5), title='Top 20 raw titles')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 3. Skill extraction sanity check

In [ ]:
from src.data_loader import load_processed
df = load_processed()
print('Processed shape:', df.shape)
df[['title','skill_count','skills']].head(5)

In [ ]:
df['skill_count'].plot.hist(bins=30, figsize=(8, 3), title='Distribution of skills extracted per posting')
plt.xlabel('# extracted skills'); plt.tight_layout(); plt.show()
print(df['skill_count'].describe())

In [ ]:
from collections import Counter
c = Counter()
for s in df['skills']: c.update(s)
top = pd.Series(dict(c.most_common(30)))
top.plot.barh(figsize=(8, 8), title='Top 30 extracted skills overall')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 4. Skill co-occurrence preview

In [ ]:
from src.skill_graph import build_skill_graph, adjacent_skills
g = build_skill_graph(df)
for skill in ['Python', 'AWS', 'React', 'SQL', 'Excel']:
    print(f'\n{skill} → top neighbours:')
    for n, w in adjacent_skills(g, skill, top_k=6):
        print(f'  {n:25s} NPMI={w:.2f}')

## 5. Role-level aggregation

In [ ]:
from src.role_aggregator import aggregate_roles
roles = aggregate_roles(df)
print('# roles:', len(roles))
roles[['title','n_postings','median_salary','top_companies']].head(15)